# eBay Electronics — Web Scraper & Data Analysis
Scrapes live electronics listings from eBay, preprocesses the data, and performs exploratory data analysis.

## 1. Imports

In [ ]:
import os
import time
import random
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from datetime import datetime
from prettytable import PrettyTable

## 2. Scrape eBay Electronics Listings

In [ ]:
# Base eBay search URL — Electronics, 240 items per page
url_pattern = (
    'https://www.ebay.com/sch/i.html?_nkw=electronics&_sacat=0&_from=R40'
    '&_trksid=p2334524.m570.l1313&_odkw=electronics&_osacat=0&_ipg=240&_pgn={page_num}'
)

In [ ]:
# Session-based scraping — mimics a real browser visit
session = requests.Session()
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Referer': 'https://www.ebay.com/',
    'DNT': '1',
}

# Visit homepage first to collect cookies
print('Initializing session...')
session.get('https://www.ebay.com', headers=headers)
time.sleep(5)

In [ ]:
product_data = []

for page_num in range(1, 6):  # Pages 1 to 5
    print(f'Scraping page {page_num}...')
    url = url_pattern.format(page_num=page_num)

    response = session.get(url, headers=headers)
    print(f'  Status: {response.status_code}')

    soup = BeautifulSoup(response.content, 'html.parser')

    # eBay's new design uses li.s-card
    items = soup.find_all('li', {'class': 's-card'})
    print(f'  Items found: {len(items)}')

    for item in items:
        # Title
        title_tag = item.find('div', {'class': 's-card__title'})
        title = title_tag.get_text(strip=True) if title_tag else ''
        if title.lower() in ['shop on ebay', '']:
            continue

        # Price
        price_tag = item.find('span', {'class': 's-card__price'})
        price_text = price_tag.get_text(strip=True).replace('$', '').replace(',', '') if price_tag else '0.0'
        if ' to ' in price_text:
            price_text = price_text.split(' to ')[0]
        try:
            price_sold = float(price_text)
        except ValueError:
            price_sold = 0.0

        # Condition
        condition_tag = item.find('div', {'class': 's-card__subtitle'})
        condition = condition_tag.get_text(strip=True) if condition_tag else ''

        # Attribute rows: listing type, shipping, location
        attr_rows = item.find_all('div', {'class': 's-card__attribute-row'})
        listing_type = 'Buy It Now'
        shipping_cost = 'Free shipping'
        item_location = ''

        for row in attr_rows:
            text = row.get_text(strip=True)
            tl = text.lower()
            if 'delivery' in tl or 'shipping' in tl:
                shipping_cost = text
            elif 'located in' in tl:
                item_location = text.replace('Located in', '').strip()
            elif 'best offer' in tl:
                listing_type = 'Best Offer'
            elif 'bid' in tl or 'auction' in tl:
                listing_type = 'Auction'

        # Seller info
        seller_name = ''
        seller_rating = ''
        secondary = item.find('div', {'class': 'su-card-container__attributes__secondary'})
        if secondary:
            spans = secondary.find_all('span', {'class': 'su-styled-text'})
            if len(spans) >= 1:
                seller_name = spans[0].get_text(strip=True)
            if len(spans) >= 2:
                seller_rating = spans[1].get_text(strip=True)

        # Link
        link_tag = item.find('a', {'class': 's-card__link'})
        link = link_tag['href'] if link_tag else ''

        product_data.append([
            title, price_sold, condition, listing_type,
            shipping_cost, item_location, seller_name, seller_rating, link
        ])

    time.sleep(random.uniform(3, 6))

print(f'\nTotal records collected: {len(product_data)}')

## 3. Preprocessing

In [ ]:
Electronics = pd.DataFrame(
    product_data,
    columns=[
        'Title', 'Price_sold', 'Condition', 'Listing_type',
        'Shipping_cost', 'Item_location', 'Seller_name', 'Seller_rating', 'Link'
    ]
)
Electronics.head(5)

In [ ]:
# Clean Condition — strip extra category text after bullet
# e.g. 'Pre-Owned ·Nintendo DS' -> 'Pre-Owned'
Electronics['Condition'] = Electronics['Condition'].str.split('·').str[0].str.strip()
Electronics['Condition'].value_counts()

In [ ]:
# Parse seller rating string into numeric columns
# Format: '100% positive (165)'
Electronics['Seller_Rating%'] = pd.to_numeric(
    Electronics['Seller_rating'].str.extract(r'([\d\.]+)%')[0], errors='coerce'
).fillna(0.0)

Electronics['Seller_feedback'] = pd.to_numeric(
    Electronics['Seller_rating'].str.extract(r'\(([\d,]+)\)')[0].str.replace(',', '', regex=False),
    errors='coerce'
).fillna(0).astype(int)

Electronics.drop(columns=['Seller_rating'], inplace=True)
Electronics[['Seller_name', 'Seller_Rating%', 'Seller_feedback']].head()

In [ ]:
# Extract numeric shipping cost and classify shipping type
Electronics['Shipping_cost_value'] = pd.to_numeric(
    Electronics['Shipping_cost'].str.extract(r'\+?\$?([\d\.]+)')[0], errors='coerce'
).fillna(0.0)

Electronics['Shipping_type'] = Electronics['Shipping_cost'].apply(
    lambda x: 'Free shipping' if 'free' in str(x).lower() else 'Paid shipping'
)

Electronics[['Shipping_cost', 'Shipping_cost_value', 'Shipping_type']].head()

In [ ]:
# Add scrape timestamp
Electronics['Scraped_date'] = datetime.today().strftime('%Y-%m-%d')
Electronics.info()

In [ ]:
Electronics.isnull().sum()

In [ ]:
# Save to CSV
output_path = os.path.join(os.getcwd(), 'Electronics.csv')
Electronics.to_csv(output_path, index=False)
print(f'Saved to: {output_path}  |  Rows: {len(Electronics)}')

## 4. Exploratory Data Analysis

In [ ]:
Electronics.describe()

In [ ]:
Electronics.nunique()

In [ ]:
numeric_data = Electronics.select_dtypes(include=['float64', 'int64'])
correlation_matrix = numeric_data.corr()
print(correlation_matrix)

### 4.1 Product Analytics

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(Electronics['Price_sold'], bins=30, color='steelblue', edgecolor='white')
plt.title('Distribution of Price — Electronics', fontsize=14)
plt.xlabel('Price (USD)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
condition_counts = Electronics['Condition'].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(x=condition_counts.values, y=condition_counts.index, palette='Blues_r')
plt.title('Electronics Listings by Condition', fontsize=14)
plt.xlabel('Number of Listings')
plt.ylabel('Condition')
plt.tight_layout()
plt.show()

In [ ]:
listing_counts = Electronics['Listing_type'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(listing_counts.values, labels=listing_counts.index, autopct='%1.1f%%',
        colors=['#4C72B0', '#DD8452', '#55A868'])
plt.title('Listing Type Distribution', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='Condition', y='Price_sold', data=Electronics, palette='Set2')
plt.title('Price Distribution by Condition', fontsize=14)
plt.xlabel('Condition')
plt.ylabel('Price (USD)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Listing_type', y='Price_sold', data=Electronics, palette='pastel')
plt.title('Price Distribution by Listing Type', fontsize=14)
plt.xlabel('Listing Type')
plt.ylabel('Price (USD)')
plt.tight_layout()
plt.show()

### 4.2 Shipping & Delivery Analysis

In [ ]:
print(Electronics['Shipping_type'].value_counts())

shipping_counts = Electronics['Shipping_type'].value_counts()
plt.figure(figsize=(7, 5))
sns.barplot(x=shipping_counts.index, y=shipping_counts.values, palette='coolwarm')
plt.title('Free vs Paid Shipping Distribution', fontsize=14)
plt.xlabel('Shipping Type')
plt.ylabel('Number of Listings')
plt.tight_layout()
plt.show()

In [ ]:
print('Total shipping cost by type:')
print(Electronics.groupby('Shipping_type')['Shipping_cost_value'].sum())
print('\nMax shipping cost by type:')
print(Electronics.groupby('Shipping_type')['Shipping_cost_value'].max())

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Shipping_type', y='Shipping_cost_value', data=Electronics, palette='coolwarm')
plt.title('Shipping Cost by Shipping Type', fontsize=14)
plt.xlabel('Shipping Type')
plt.ylabel('Shipping Cost (USD)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(Electronics['Price_sold'], Electronics['Shipping_cost_value'],
            alpha=0.4, color='teal', edgecolors='none')
plt.title('Relationship between Price and Shipping Cost', fontsize=14)
plt.xlabel('Price (USD)')
plt.ylabel('Shipping Cost (USD)')
plt.tight_layout()
plt.show()

In [ ]:
location_counts = Electronics['Item_location'].value_counts().head(15)
if not location_counts.empty and location_counts.sum() > 0:
    plt.figure(figsize=(10, 6))
    sns.barplot(x=location_counts.values, y=location_counts.index, palette='viridis')
    plt.title('Top 15 Seller Locations', fontsize=14)
    plt.xlabel('Number of Listings')
    plt.ylabel('Location')
    plt.tight_layout()
    plt.show()
else:
    print('No location data available.')

### 4.3 Seller Analysis

In [ ]:
if Electronics['Seller_feedback'].sum() > 0:
    top_sellers = (
        Electronics.groupby('Seller_name')['Seller_feedback']
        .max().nlargest(10).reset_index()
    )
    plt.figure(figsize=(10, 5))
    sns.barplot(x='Seller_feedback', y='Seller_name', data=top_sellers, palette='Blues_r')
    plt.title('Top 10 Sellers by Feedback Count', fontsize=14)
    plt.xlabel('Feedback Count')
    plt.ylabel('Seller Name')
    plt.tight_layout()
    plt.show()
else:
    print('Seller feedback data not available.')

In [ ]:
if Electronics['Seller_Rating%'].sum() > 0:
    plt.figure(figsize=(10, 6))
    plt.scatter(Electronics['Seller_Rating%'], Electronics['Price_sold'],
                alpha=0.4, color='purple', edgecolors='none')
    plt.title('Seller Rating vs Price', fontsize=14)
    plt.xlabel('Seller Rating (%)')
    plt.ylabel('Price (USD)')
    plt.tight_layout()
    plt.show()
else:
    print('Seller rating data not available.')

In [ ]:
top_products = Electronics.groupby('Title')['Price_sold'].max().nlargest(10)

table = PrettyTable()
table.field_names = ['Title', 'Price (USD)']
table.align['Title'] = 'l'

for title, price in zip(top_products.index, top_products.values):
    table.add_row([title[:65], f'${price:.2f}'])

print(table)